In [28]:
import os
from torchvision.io import decode_image
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class SegmentationDataset(Dataset):
    def __init__(self, mask_dir, img_dir, transform=None):
        self.mask_dir = mask_dir
        self.img_dir = img_dir
        self.transform = transform
        self.image_paths = []
        self.mask_paths = []

        img_names=os.listdir(img_dir)
        for img_name in img_names:
            base_name=img_name.replace('.jpg', '')
            mask_name = f"{base_name}_segmentation.png"
            img_path = os.path.join(self.img_dir, img_name)
            mask_path = os.path.join(self.mask_dir, mask_name)
            if os.path.exists(mask_path):
                self.image_paths.append(img_path)
                self.mask_paths.append(mask_path)
            else:
                print(f"Pominięto: Brak maski dla obrazu {img_name}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(img_path).convert("RGB")
        image = np.array(image)
        mask = Image.open(mask_path).convert("L")
        mask = np.array(mask)
        mask = mask.astype(np.float32) / 255.0

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        
        if mask.ndimension() == 2:
            mask = mask.unsqueeze(0)
        
        return image, mask

In [29]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.6),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

test_val_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [30]:
trainSet=SegmentationDataset("SegDataset/train/trainGt", "SegDataset/train/trainInput", transform=train_transform)
valSet=SegmentationDataset("SegDataset/val/valGt", "SegDataset/val/valInput", transform=test_val_transform)
testSet=SegmentationDataset("SegDataset/test/testGt", "SegDataset/test/testInput", transform=test_val_transform)

In [31]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(trainSet, batch_size=32, shuffle=True)
test_dataloader = DataLoader(testSet, batch_size=32, shuffle=False)
val_dataloader = DataLoader(valSet, batch_size=32, shuffle=False)

In [32]:
import segmentation_models_pytorch as smp

#Used to be Unetplusplus but it was too much load for my GPU
model = smp.UnetPlusPlus(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
)

print(f"Pomyślnie załadowano model U-Net++ z koderem {model.name}")

Pomyślnie załadowano model U-Net++ z koderem unetplusplus-resnet34


In [33]:
import torch

#Changing learning rates for each part of the model
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 1e-5},
    {'params': model.decoder.parameters(), 'lr': 1e-3},
    {'params': model.segmentation_head.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

In [34]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.1, 
    patience=5, 
)

In [35]:
bce_criterion = torch.nn.BCEWithLogitsLoss()
dice_criterion = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA jest dostępna. Używam GPU.")
else:
    device = torch.device("cpu")
    print("GPU niedostępne. Przełączono na CPU.")

model.to(device)

CUDA jest dostępna. Używam GPU.


UnetPlusPlus(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, ep

In [36]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0, verbose=False):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save(model.state_dict(), 'checkpoint.pt')
        self.val_loss_min = val_loss

In [37]:
from tqdm.auto import tqdm
NUM_EPOCHS = 50
early_stopping = EarlyStopping(patience=10, verbose=True)

for epoch in range(NUM_EPOCHS):
    print(f"\n--- EPOKA {epoch+1}/{NUM_EPOCHS} ---")
    
    # ================= FAZA TRENINGU =================
    model.train()
    train_loss = 0.0
    train_bar = tqdm(train_dataloader, desc="Trening", leave=True)
    
    for images, masks in train_bar:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        
        # Obliczanie straty komponentowej
        loss = bce_criterion(logits, masks) + dice_criterion(logits, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    avg_train_loss = train_loss / len(train_dataloader)
    print(f"Training Loss: {avg_train_loss:.4f}")
    
    # ================= FAZA WALIDACJI =================
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    
    with torch.no_grad():
        for images, masks in val_dataloader:
            images, masks = images.to(device), masks.to(device)
            
            logits = model(images)
            # Poprawiono: Użycie właściwych funkcji kosztu zamiast niezdefiniowanego 'criterion'
            loss = bce_criterion(logits, masks) + dice_criterion(logits, masks)
            val_loss += loss.item()
            
            # Obliczanie metryki Dice
            preds = (torch.sigmoid(logits) > 0.5).float()
            intersection = (preds * masks).sum()
            dice = (2. * intersection) / (preds.sum() + masks.sum() + 1e-8)
            val_dice += dice.item()
        
    avg_val_loss = val_loss / len(val_dataloader)
    avg_val_dice = val_dice / len(val_dataloader)
    print(f"Validation Loss: {avg_val_loss:.4f} | Dice Score: {avg_val_dice * 100:.2f}%")
    
    # Aktualizacja harmonogramu uczenia i Early Stopping (raz na epokę!)
    scheduler.step(avg_val_loss)
    early_stopping(avg_val_loss, model)
    
    if early_stopping.early_stop:
        print("Early stopping triggered. Przerywam trening.")
        break


--- EPOKA 1/50 ---


Trening: 100%|██████████| 65/65 [02:28<00:00,  2.29s/it, loss=0.2468]


Training Loss: 0.3711
Validation Loss: 0.2602 | Dice Score: 93.15%
Validation loss decreased (inf --> 0.260164). Saving model...

--- EPOKA 2/50 ---


Trening: 100%|██████████| 65/65 [02:20<00:00,  2.16s/it, loss=0.2639]


Training Loss: 0.2763
Validation Loss: 0.2399 | Dice Score: 93.68%
Validation loss decreased (0.260164 --> 0.239935). Saving model...

--- EPOKA 3/50 ---


...

Training Loss: 0.1744
Validation Loss: 0.2084 | Dice Score: 94.57%
EarlyStopping counter: 8 out of 10

--- EPOKA 21/50 ---


Trening: 100%|██████████| 65/65 [02:20<00:00,  2.17s/it, loss=0.1647]


Training Loss: 0.1737
Validation Loss: 0.2115 | Dice Score: 94.49%
EarlyStopping counter: 9 out of 10

--- EPOKA 22/50 ---


Trening: 100%|██████████| 65/65 [02:21<00:00,  2.17s/it, loss=0.1785]


Training Loss: 0.1701
Validation Loss: 0.2099 | Dice Score: 94.51%
EarlyStopping counter: 10 out of 10
Early stopping triggered. Przerywam trening.
